# Turn production failures into a prompt regression check

Ridgeline Outfitters routes messages to a returns queue or a repair queue. The application needs the exact queue name, but its prompt sometimes produces an explanation. Use synthetic examples of logged failures to optimize that prompt, then compare the selected prompt with the logged replies on tickets the optimizer never saw. Nothing is deployed. Model calls use your OpenAI account; scoring runs locally.

## Install Agent Learning Kit

In [ ]:
%pip install -q agent-learning-kit==0.3.7

In [ ]:
import os
from getpass import getpass

# Exported variables work in Jupyter; otherwise enter the keys at the hidden prompts.
for name in ['OPENAI_API_KEY']:
    if not os.environ.get(name):
        value = getpass(f"{name}: ")
        if not value:
            raise ValueError(f"Missing {name}")
        os.environ[name] = value
        del value

## Split the regression cases before training

Each row contains the customer’s ticket, the queue a reviewer expected, and a representative logged reply. Keep real customer data out of model calls unless you have permission to send it. The training rows and held-out rows each cover both queues.

In [ ]:
train = [
    {"ticket": "Can I return an unused camp stove?", "queue": "RETURNS", "logged": "Contact our returns team."},
    {"ticket": "Can you fix a torn tent seam?", "queue": "REPAIRS", "logged": "Try our repair desk."},
    {"ticket": "I want a refund on this unopened bottle.", "queue": "RETURNS", "logged": "Please send it back."},
    {"ticket": "The buckle on my old pack is cracked. Can you mend it?", "queue": "REPAIRS", "logged": "Ask about repairs."},
]
held_out = [
    {"ticket": "These unworn boots don't fit. Where can I send them back?", "queue": "RETURNS", "logged": "The returns team can help."},
    {"ticket": "Could someone replace the zipper on my sleeping bag?", "queue": "REPAIRS", "logged": "Talk to the repair team."},
]
assert not {r['ticket'] for r in train} & {r['ticket'] for r in held_out}

## Optimize against the same exact-match rule

The teacher sees scored training examples and proposes a rewrite. Two rounds score the initial prompt and its first proposed rewrite on the same four tickets. The held-out tickets and their replies are not passed to the optimizer.

In [ ]:
from fi.evals.metrics import Equals
from fi.opt.base import Evaluator
from fi.opt.datamappers import BasicDataMapper
from fi.opt.generators import LiteLLMGenerator
from fi.opt.optimizers import MetaPromptOptimizer

model = "openai/gpt-4o-mini"
seed = "Explain whether RETURNS or REPAIRS should handle this ticket: {ticket}"
evaluator = Evaluator(metric=Equals(config={"case_sensitive": True}))
mapper = BasicDataMapper({"response": "generated_output", "expected_response": "queue"})
optimizer = MetaPromptOptimizer(teacher_generator=LiteLLMGenerator(model, "{prompt}"), task_model=model)
result = optimizer.optimize(
    evaluator=evaluator, data_mapper=mapper,
    dataset=[{"ticket": r["ticket"], "queue": r["queue"]} for r in train],
    initial_prompts=[seed], num_rounds=2, eval_subset_size=len(train),
    task_description=("Return only RETURNS for refunds or REPAIRS for fixes. "
                      "Use {ticket} as the only placeholder."),
)
if len(result.history) != 2:
    raise RuntimeError("Both prompts must be scored before comparing them")
for i, round_ in enumerate(result.history, 1):
    print(f"Round {i}: {round_.average_score:.0%} training exact match")
selected = result.best_generator.get_prompt_template()
print("Selected prompt:", selected)

## Replay held-out tickets and compare the actual replies

Score the logged reply and the selected prompt’s fresh reply with the same case-sensitive metric. A pass means the entire string matches the routing label.

In [ ]:
from string import Formatter

if {field for _, field, _, _ in Formatter().parse(selected) if field is not None} != {"ticket"}:
    raise ValueError("Selected prompt must take only the ticket field")

def score(reply, row):
    if not reply:
        raise RuntimeError("Empty reply is not a valid generation")
    [verdict] = evaluator.evaluate([{"response": reply, "expected_response": row["queue"]}])
    return verdict.score

old_passes = new_passes = 0
for row in held_out:
    reply = result.best_generator.generate({"ticket": row["ticket"]})
    old_passes += score(row["logged"], row)
    new_passes += score(reply, row)
    print(f"{row['ticket']}\n  Expected: {row['queue']} | Logged: {row['logged']!r} | New: {reply!r}")
print(f"Held-out exact matches: logged {old_passes:.0f}/{len(held_out)}, selected {new_passes:.0f}/{len(held_out)}")

## Decide what to do next

Keep a per-ticket record, even when the total goes up: one newly broken case can hide behind an improvement elsewhere. A two-ticket holdout demonstrates the loop, not a production success rate. Before promotion, evaluate a larger, representative holdout including previously successful tickets.